# User and expert evaluation
Analyse the supplied evaluation forms. Keep respondent records and reviewer identities private. Export aggregate results for the thesis. Self-reported task completion is reported separately from comprehension answers and observed browser tests.


In [1]:
from pathlib import Path
import json
import re
import hashlib
import numpy as np
import pandas as pd
from IPython.display import display

try:
    from google.colab import drive
except ImportError:
    ROOT = Path.cwd().resolve()
    if ROOT.name == 'notebooks':
        ROOT = ROOT.parent
else:
    drive.mount('/content/drive')
    ROOT = Path('/content/drive/MyDrive/Final Research - C D Jayasinghe')

PRIVATE = ROOT / 'outputs/evaluation_private'
REPORT = ROOT / ('evaluation' if (ROOT / '.git').exists() else 'outputs')
PRIVATE.mkdir(parents=True, exist_ok=True)
REPORT.mkdir(parents=True, exist_ok=True)

def locate(name):
    candidates = [ROOT/'data/evaluation'/name, ROOT/'data'/name,
                  ROOT/name, ROOT/name/name, ROOT.parent/name/name]
    matches = [p for p in candidates if p.is_file()]
    if not matches:
        raise FileNotFoundError(f'Place {name} in the project data/evaluation folder.')
    return matches[0]

USER_FILE = locate('Pathways DSS User Evaluation.csv')
EXPERT_FILE = locate('Pathways DSS Expert Review.csv')
users = pd.read_csv(USER_FILE, dtype=str, keep_default_na=False, encoding='utf-8-sig')
experts = pd.read_csv(EXPERT_FILE, dtype=str, keep_default_na=False, encoding='utf-8-sig')
if users.empty or experts.empty:
    raise ValueError('Both evaluation files must contain responses.')
users = users.apply(lambda column: column.str.strip())
experts = experts.apply(lambda column: column.str.strip())

def column(frame, prefix):
    matches = [c for c in frame if c.startswith(prefix)]
    if len(matches) != 1:
        raise ValueError(f'Expected one column beginning {prefix!r}; found {len(matches)}.')
    return matches[0]

def save(frame, name):
    frame.to_csv(REPORT/name, index=False, float_format='%.6f')

consent_text = ('I am 18 or older, am a Sri Lankan university student, have read '
                'the information above, and voluntarily agree to participate.')
consent = users[column(users, 'CONSENT.')].eq(consent_text)
submitted_users = len(users)
excluded_no_consent = int((~consent).sum())
users = users.loc[consent].reset_index(drop=True)
if users.empty:
    raise ValueError('No consenting eligible user responses.')
print(f'User records: {submitted_users}; eligible consenting records: {len(users)}; expert records: {len(experts)}')

User records: 10; eligible consenting records: 10; expert records: 2


## 1. System Usability Scale
Odd items contribute response minus 1; even items contribute 5 minus response. Multiply the sum by 2.5. Report scores on a 0–100 scale, not as a percentage. Exclude incomplete SUS records without imputation.


In [2]:
sus_columns = [column(users, f'SUS{i}.') for i in range(1, 11)]
sus = users[sus_columns].replace('', np.nan).apply(pd.to_numeric, errors='raise')
if (sus.notna() & ~sus.isin([1,2,3,4,5])).any().any():
    raise ValueError('SUS responses must be integers from 1 to 5.')
complete_sus = sus.dropna()
if complete_sus.empty:
    raise ValueError('No complete SUS records.')
scores = ((complete_sus.iloc[:, ::2]-1).sum(axis=1)
          + (5-complete_sus.iloc[:, 1::2]).sum(axis=1))*2.5
sus_summary = pd.DataFrame([{
    'submitted_n': submitted_users, 'eligible_consenting_n': len(users),
    'complete_n': len(scores), 'excluded_incomplete_n': len(sus)-len(scores),
    'mean_sus': scores.mean(), 'sd_sus': scores.std(ddof=1),
    'median_sus': scores.median(), 'minimum_sus': scores.min(), 'maximum_sus': scores.max()
}])
item_rows = []
for i, c in enumerate(sus_columns, 1):
    values = sus[c].dropna()
    item_rows.append({'item':f'SUS{i}', 'statement':c.split('. ',1)[1],
                      'n':len(values), 'mean_raw_rating':values.mean(),
                      'sd_raw_rating':values.std(ddof=1),
                      **{f'rating_{v}_n':int(values.eq(v).sum()) for v in range(1,6)}})
sus_item_summary = pd.DataFrame(item_rows)
save(sus_summary, 'sus_summary.csv')
save(sus_item_summary, 'sus_item_summary.csv')
private_sus = sus.copy()
private_sus.columns = [f'SUS{i}' for i in range(1,11)]
private_sus.insert(0, 'participant_id', [f'U{i:02}' for i in range(1,len(users)+1)])
private_sus['sus_score'] = scores
private_sus.to_csv(PRIVATE/'sus_scoring.csv', index=False)
display(sus_summary)

,submitted_n,eligible_consenting_n,complete_n,excluded_incomplete_n,mean_sus,sd_sus,median_sus,minimum_sus,maximum_sus
0,10,10,10,0,91.25,7.569126,91.25,75.0,100.0


## 2. Comprehension and self-reported task completion
Use the fixed synthetic case: 20 Rarely responses give total 20/80 and each category 4/16; skipping D01 makes total and Coercion incomplete. The researcher confirmed that C2 values exported as “Apr-16” originally represented “4/16”. Preserve that correction in the audit and leave source files unchanged. Do not infer task timings or independently observed success from questionnaire responses.


In [3]:
# Confirmed directly by the researcher in this conversation.
C2_DATE_CONVERSION_CONFIRMED = True
IDENTICAL_ROWS_CONFIRMED_DIFFERENT_PARTICIPANTS = True
answers = {
    'C1': '20 / 80',
    'C2': '4/16',
    'C3': 'Both scores are incomplete; the app shows observed subtotals rather than complete scores.',
    'C4': 'The student’s stated willingness to seek professional help.',
    'C5': 'Mithuru Piyasa / Natpu Nilayam',
    'C6': 'Shows a possible next step and conversation prompt; the user decides whether to contact anyone.'
}
def normalise(value):
    return re.sub(r'\s+', ' ', value.replace('’', "'")).strip().casefold()

comprehension_rows = []
c2_corrected_n = 0
for item, expected in answers.items():
    c = column(users, f'{item}.')
    responses = users[c].copy()
    unresolved = pd.Series(False, index=responses.index)
    if item == 'C2':
        converted = responses.eq('Apr-16')
        if C2_DATE_CONVERSION_CONFIRMED:
            c2_corrected_n = int(converted.sum())
            responses.loc[converted] = '4/16'
        else:
            unresolved = converted
        responses = responses.str.replace(' ', '', regex=False)
    answered = responses.ne('') & ~unresolved
    correct = answered & responses.map(normalise).eq(normalise(expected))
    comprehension_rows.append({
        'item':item, 'question':c.split('. ',1)[1], 'expected_answer':expected,
        'eligible_n':len(users), 'scorable_n':int(answered.sum()),
        'missing_n':int(responses.eq('').sum()), 'unresolved_n':int(unresolved.sum()),
        'correct_n':int(correct.sum()),
        'correct_percent_of_scorable':100*correct.sum()/answered.sum() if answered.any() else np.nan
    })
comprehension = pd.DataFrame(comprehension_rows)
task_rows = []
for task in range(1,6):
    matches = [c for c in users if c.startswith('TASKS.') and f'[T{task} ' in c]
    if len(matches) != 1:
        raise ValueError(f'Expected one T{task} self-report column.')
    counts = users[matches[0]].value_counts(dropna=False)
    for outcome, count in counts.items():
        task_rows.append({'task':f'T{task}', 'description':matches[0].split('[',1)[1].rstrip(']'),
                          'reported_outcome':outcome or 'Missing', 'n':int(count),
                          'eligible_n':len(users), 'percent':100*count/len(users),
                          'measurement':'Participant self-report; not independently timed or observed'})
tasks = pd.DataFrame(task_rows)
devices = users[column(users,'DEVICE.')].replace('', 'Missing').value_counts().rename_axis('device').reset_index(name='n')
devices['percent'] = 100*devices['n']/len(users)
save(comprehension, 'user_comprehension_summary.csv')
save(tasks, 'user_task_summary.csv')
save(devices, 'user_device_summary.csv')
display(comprehension[['item','eligible_n','scorable_n','correct_n','correct_percent_of_scorable']])
display(tasks[['task','reported_outcome','n','percent']])

,item,eligible_n,scorable_n,correct_n,correct_percent_of_scorable
0,C1,10,10,10,100.0
1,C2,10,10,10,100.0
2,C3,10,10,10,100.0
3,C4,10,10,10,100.0
4,C5,10,10,10,100.0
5,C6,10,10,10,100.0


,task,reported_outcome,n,percent
0,T1,Completed independently,10,100.0
1,T2,Completed independently,10,100.0
2,T3,Completed independently,10,100.0
3,T4,Completed independently,10,100.0
4,T5,Completed independently,10,100.0


## 3. Expert review
Summarise the six criteria and overall recommendation descriptively. Exclude “Unable to assess” ratings from criterion means and report their counts. The supplied reviewers are domain professionals; do not label this as a technical audit or clinical validation. Names and qualifications are not exported.


In [4]:
domain_areas = {'Psychology / clinical psychology', 'Counselling'}
if not experts[column(experts,'EX1.')].isin(domain_areas).all():
    raise ValueError('Review professional-area mapping before adding reviewers from other disciplines.')
criterion_columns = [c for c in experts if c.startswith('EX3.')]
if len(criterion_columns) != 6:
    raise ValueError('Expected six expert review criteria.')
expert_rows = []
private_ratings = []
for number, c in enumerate(criterion_columns,1):
    raw = experts[c]
    unable = raw.str.casefold().eq('unable to assess')
    missing = raw.eq('')
    rating = pd.to_numeric(raw.str.extract(r'^([1-5])(?:\s|$)', expand=False), errors='coerce')
    if (~missing & ~unable & rating.isna()).any():
        raise ValueError(f'Unknown expert rating in criterion {number}.')
    valid = rating.dropna()
    criterion = c.split('[',1)[1].rstrip(']')
    expert_rows.append({'criterion_id':f'EX3-{number}', 'criterion':criterion,
                        'reviewer_group':'domain', 'submitted_n':len(experts),
                        'rated_n':len(valid), 'unable_to_assess_n':int(unable.sum()),
                        'missing_n':int(missing.sum()), 'mean_rating':valid.mean(),
                        'sd_rating':valid.std(ddof=1), 'median_rating':valid.median(),
                        'min_rating':valid.min(), 'max_rating':valid.max(),
                        **{f'rating_{v}_n':int(valid.eq(v).sum()) for v in range(1,6)}})
    for index, value in rating.items():
        private_ratings.append({'reviewer_id':f'E{index+1:02}', 'criterion_id':f'EX3-{number}',
                                'rating':value})
expert_summary = pd.DataFrame(expert_rows)
recommendations = experts[column(experts,'EX4.')].replace('', 'Missing').value_counts().rename_axis('recommendation').reset_index(name='n')
recommendations['percent'] = 100*recommendations['n']/len(experts)
save(expert_summary, 'expert_summary.csv')
save(recommendations, 'expert_recommendations.csv')
pd.DataFrame(private_ratings).to_csv(PRIVATE/'expert_ratings.csv',index=False)
display(expert_summary[['criterion_id','rated_n','mean_rating','sd_rating','unable_to_assess_n']])
display(recommendations)

,criterion_id,rated_n,mean_rating,sd_rating,unable_to_assess_n
0,EX3-1,2,5.0,0.0,0
1,EX3-2,2,5.0,0.0,0
2,EX3-3,2,5.0,0.0,0
3,EX3-4,2,5.0,0.0,0
4,EX3-5,2,5.0,0.0,0
5,EX3-6,2,5.0,0.0,0


,recommendation,n,percent
0,Suitable for research use without changes,2,100.0


## 4. Data checks and interpretation
Report the sample sizes and collection limitations alongside the results. Questionnaire answers support perceived usability and comprehension; they do not establish clinical safety, improved help-seeking, accessibility conformance or load performance.


In [5]:
def substantive(series):
    return series.map(normalise).map(lambda value: bool(re.search(r'[a-z0-9]', value)))

positive_feedback = users[column(users,'FEEDBACK1.')]
improvement_feedback = users[column(users,'FEEDBACK2.')]
expert_revisions = experts[column(experts,'EX5.')]
audit = {
    'user_source':USER_FILE.name, 'expert_source':EXPERT_FILE.name,
    'user_source_sha256':hashlib.sha256(USER_FILE.read_bytes()).hexdigest(),
    'expert_source_sha256':hashlib.sha256(EXPERT_FILE.read_bytes()).hexdigest(),
    'user_submitted_n':submitted_users, 'user_eligible_consenting_n':len(users),
    'user_excluded_no_consent_n':excluded_no_consent,
    'sus_complete_n':len(scores), 'sus_excluded_incomplete_n':len(sus)-len(scores),
    'expert_n':len(experts), 'expert_group':'domain', 'technical_reviewer_n':0,
    'additional_identical_user_rows':int(users.duplicated(keep='first').sum()),
    'identical_rows_confirmed_different_participants':IDENTICAL_ROWS_CONFIRMED_DIFFERENT_PARTICIPANTS,
    'duplicate_identity_check':'Researcher confirmed the identical rows represent different participants; all records retained. No stable respondent identifiers are present for independent verification.',
    'c2_correction':{'source_value':'Apr-16', 'analysis_value':'4/16',
                     'affected_n':c2_corrected_n, 'confirmed_by_researcher':C2_DATE_CONVERSION_CONFIRMED,
                     'source_file_modified':False},
    'user_substantive_positive_feedback_n':int(substantive(positive_feedback).sum()),
    'user_nonplaceholder_improvement_feedback_n':int(substantive(improvement_feedback).sum()),
    'expert_substantive_revision_comment_n':int(substantive(expert_revisions).sum()),
    'limitations':[
        f'Small supplied sample: {len(users)} user records and {len(experts)} domain expert records; no population-level or clinical validation claim.',
        'Task completion is self-reported. Completion times, observed assistance and observed critical-error records were not supplied.',
        'The CSV exports do not establish sampling method, response rate, collection dates or the exact evaluated deployment.',
        'All comprehension answers and expert ratings are uniform in this supplied sample; report this ceiling effect.',
        'SUS is a 0-100 score, not a percentage. Expert ratings are descriptive reviewer judgements.',
        'Source records, reviewer identities and row-level derived scores are excluded from Git-tracked aggregates.'
    ]
}
(REPORT/'user_expert_data_audit.json').write_text(json.dumps(audit,indent=2),encoding='utf-8')
assert scores.between(0,100).all()
assert (sus_item_summary[[f'rating_{v}_n' for v in range(1,6)]].sum(axis=1).to_numpy() == sus.notna().sum().to_numpy()).all()
assert (comprehension['correct_n'] <= comprehension['scorable_n']).all()
assert tasks.groupby('task')['n'].sum().eq(len(users)).all()
assert expert_summary['rated_n'].add(expert_summary['unable_to_assess_n']).add(expert_summary['missing_n']).eq(len(experts)).all()
print('Aggregate evaluation results saved. Source CSVs unchanged; row-level results saved privately.')

Aggregate evaluation results saved. Source CSVs unchanged; row-level results saved privately.


## 5. Optional observed decision-support tasks
Only analyse `outputs/decision_support_tasks.csv` when independently observed sessions are available. Expected columns: `participant_id`, `task`, `completed`, `correct`, `seconds`, `critical_error`. Use task names `score_reading`, `missing_score`, `intention_distinction`, `find_service`, `choose_step`. Do not manufacture these records from self-reported form answers.


In [6]:
observed_path = ROOT/'outputs/decision_support_tasks.csv'
if observed_path.is_file():
    observed = pd.read_csv(observed_path)
    required = {'participant_id','task','completed','correct','seconds','critical_error'}
    allowed = {'score_reading','missing_score','intention_distinction','find_service','choose_step'}
    if not required.issubset(observed.columns) or observed.empty:
        raise ValueError('Missing observed task columns or records.')
    if observed[['participant_id','task']].isna().any().any() or observed.duplicated(['participant_id','task']).any():
        raise ValueError('Missing identifiers or duplicate observed task records.')
    if not observed['task'].isin(allowed).all():
        raise ValueError('Unexpected observed task name.')
    for c in ['completed','correct','critical_error']:
        if not observed[c].isin([0,1]).all():
            raise ValueError('Observed task indicators must be 0 or 1.')
    observed['seconds'] = pd.to_numeric(observed['seconds'],errors='raise')
    if not np.isfinite(observed['seconds']).all() or observed['seconds'].lt(0).any():
        raise ValueError('Observed times must be finite and nonnegative.')
    if observed['correct'].gt(observed['completed']).any():
        raise ValueError('Correct tasks must have been completed.')
    observed_summary = observed.groupby('task').agg(
        n=('participant_id','count'),completion_rate=('completed','mean'),
        correct_rate=('correct','mean'),median_seconds=('seconds','median'),
        critical_errors=('critical_error','sum')).reset_index()
    save(observed_summary, 'decision_support_observed_summary.csv')
    display(observed_summary)
else:
    print('No independently observed or timed task records supplied; questionnaire self-reports are summarised separately.')

No independently observed or timed task records supplied; questionnaire self-reports are summarised separately.
